In [60]:
from pathlib import Path

import kagglehub
import pandas as pd
import os
from os import listdir
from os.path import isfile, join
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
from torchvision.transforms import Compose, ToTensor

from src.helpers import print_data_folder_structure, plot_img, visual_exploration


In [61]:
dataset_path = kagglehub.competition_download('cifar-10')
labels_path = os.path.join(dataset_path, 'trainLabels.csv')
train_path = os.path.join(dataset_path, 'train')

In [62]:
print_data_folder_structure(dataset_path, 1)

cifar-10/
├── sampleSubmission.csv
├── test.7z
├── train.7z
├── trainLabels.csv
├── test/
└── train/


In [63]:
labels_df = pd.read_csv(labels_path)

In [64]:
files = [(int(Path(join(train_path, f)).stem), f) for f in listdir(train_path) if isfile(join(train_path, f))]

In [65]:
files_df = pd.DataFrame(files, columns=["id", "file_name"])

In [66]:
cifar_pd = pd.merge(labels_df, files_df, on='id', how='inner')

In [67]:
all_labels = cifar_pd['label'].drop_duplicates().reset_index()
all_labels['target'] = all_labels.index
all_labels = all_labels.drop(columns=['index'])

In [68]:
cifar_pd = pd.merge(cifar_pd, all_labels, on='label', how='left')

In [69]:
cifar_pd

,id,label,file_name,target
0,1,frog,1.png,0
1,2,truck,2.png,1
2,3,truck,3.png,1
3,4,deer,4.png,2
4,5,automobile,5.png,3
...,...,...,...,...
49995,49996,bird,49996.png,4
49996,49997,frog,49997.png,0
49997,49998,truck,49998.png,1
49998,49999,automobile,49999.png,3


In [70]:
all_target_labels = [
    'frog',
    'bird',
    'ship',
    'cat'
]
filtered_cifar = cifar_pd[cifar_pd['label'].isin(all_target_labels)]

In [71]:
class TmpCifarDataset(Dataset):
    def __init__(self, train_lib_dir, df, transform=None):
        self.train_lib_dir = train_lib_dir
        self.df = df
        self.transform = transform
        self.labels = df['label']
        self.transform = Compose([
            ToTensor()
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        item = self.df.iloc[idx]
        image = self.retrieve_image(item)
        if self.transform is not None:
            image = self.transform(image)
        return image, item['target']

    def retrieve_image(self, item):
        image_path = os.path.join(self.train_lib_dir, item['file_name'])
        with Image.open(image_path) as img:
            image = img.convert("RGB")
        return image

    def get_label_description(self, idx):
        return self.df.iloc[idx]['label']

In [72]:
dataset = TmpCifarDataset(train_path, filtered_cifar)
full_dataset = TmpCifarDataset(train_path, cifar_pd)

In [82]:
print(f'Length of the dataset: {len(full_dataset)}')

Length of the dataset: 50000


In [88]:
# checking unique classes and amount of samples per class
print(f'Count of unique classes: {len(full_dataset.labels.value_counts())}')
full_dataset.labels.value_counts().sort_index()

Count of unique classes: 10


label
airplane      5000
automobile    5000
bird          5000
cat           5000
deer          5000
dog           5000
frog          5000
horse         5000
ship          5000
truck         5000
Name: count, dtype: int64

10

In [76]:
datasets = random_split(dataset, [0.7, 0.15, 0.15])
train_data_loader = DataLoader(datasets[0], batch_size=64, shuffle=True)
val_data_loader = DataLoader(datasets[1], batch_size=len(datasets[1]), shuffle=False)
test_data_loader = DataLoader(datasets[2], batch_size=len(datasets[2]), shuffle=False)

In [ ]:
# Checking shapes and channels
for images, labels in test_data_loader:
    assert images.ndim == 4
    assert images.shape[1:] == (3, 32, 32)
    assert images.shape[0] == labels.shape[0]

In [ ]:
# Look at a sample to check it's working correctly
sel_idx = 2
img, label = dataset[sel_idx]

# Visualize the image
plot_img(img)

# Print its description
print(f'Description: {dataset.get_label_description(sel_idx)}')

# Print its shape
print(f'Image shape: {img.size}\n')  # PIL image size is (width, height)

In [ ]:
visual_exploration(full_dataset, num_rows=2, num_cols=4)